In [1]:
import pandas as pd
import numpy as np
# --- Tabela 1: autorzy ---
autorzy = pd.DataFrame({
    'autor_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'imie': ['Olga', 'Stanisław', 'Andrzej', 'Wisława', 'Ryszard',
             'Dorota', 'Szczepan', 'Jacek'],
    'nazwisko': ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuściński',
    'Masłowska', 'Twardoch', 'Dehnel'],
    'kraj': ['Polska', 'Polska', 'Polska', 'Polska', 'Polska',
    'Polska', 'Polska', 'Polska'],
    'nagrody': ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter', 'Polityka', 'NIKE', 'NIKE']
})
# --- Tabela 2: książki ---
ksiazki = pd.DataFrame({
    'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    'autor_id': [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
    'tytul': ['Księgi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada',
              'Wiedźmin', 'Narrenturm', 'Wiersze wybrane', 'Podróże z Herodotem',
              'Wojna polsko-ruska', 'Morfina', 'Król', 'Lala'],
    'kategoria': ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi',
                  'fantasy', 'fantasy', 'poezja', 'reportaz',
                  'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
    'cena': [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00, 52.00, 38.00, 48.00, 59.00, 44.00],
    'strony': [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})
# --- Tabela 3: zamówienia ---
np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
    'zam_id': range(5001, 5001 + n),
    'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
    'ilosc': np.random.randint(1, 6, size=n),
    'data': pd.to_datetime('2026-01-01') +
pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
    'kanal': np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=
[0.6, 0.3, 0.1]),
    'miasto': np.random.choice(['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk',
'Poznań'], size=n)
})
print(f"Autorzy: {autorzy.shape}")
print(f"Ksiazki: {ksiazki.shape}")
print(f"Zamowienia: {zamowienia.shape}")

Autorzy: (8, 5)
Ksiazki: (12, 6)
Zamowienia: (80, 6)


Ćwiczenie 1: merge() — łączenie dwóch tabel (20 min)

In [2]:
# Połącz ksiazki + autorzy po kluczu autor_id
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')
print(f"Shape: {ksiazki_z_autorami.shape}")
ksiazki_z_autorami.head()

Shape: (12, 10)


,ksiazka_id,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,101,1,Księgi Jakubowe,historyczna,79.9,912,Olga,Tokarczuk,Polska,Nobel
1,102,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
2,103,2,Solaris,sci-fi,39.0,198,Stanisław,Lem,Polska,SFF
3,104,2,Cyberiada,sci-fi,42.0,295,Stanisław,Lem,Polska,SFF
4,105,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


Zadanie 1a: Odpowiedz w komórce Markdown:
Ile wierszy ma wynik?
12
Ile kolumn?
10
Jakie nowe kolumny pojawiły się (z tabeli autorzy)?
imie, nazwisko, kraj, nagrody

Zadanie 1b: Porównaj cztery typy złączenia:

In [3]:
ksiazki_test = pd.concat([ 
    ksiazki, 
    pd.DataFrame({'ksiazka_id': [999], 'autor_id': [999], 'tytul': ['Ksiazka bez autora'], 'kategoria': ['test'], 'cena': [30.0], 'strony':[100]}) 
], ignore_index=True) 

inner = ksiazki_test.merge(autorzy, on='autor_id', how='inner') 
left = ksiazki_test.merge(autorzy, on='autor_id', how='left') 
right = ksiazki_test.merge(autorzy, on='autor_id', how='right') 
outer = ksiazki_test.merge(autorzy, on='autor_id', how='outer') 

print(f"inner: {inner.shape[0]} wierszy") 
print(f"left: {left.shape[0]} wierszy") 
print(f"right: {right.shape[0]} wierszy") 
print(f"outer: {outer.shape[0]} wierszy") 

audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True) 
print("\nAudyt złączenia:")
print(audyt['_merge'].value_counts()) 

inner: 12 wierszy
left: 13 wierszy
right: 12 wierszy
outer: 13 wierszy

Audyt złączenia:
_merge
both          12
left_only      1
right_only     0
Name: count, dtype: int64


In [4]:
audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True)
audyt['_merge'].value_counts()

_merge
both          12
left_only      1
right_only     0
Name: count, dtype: int64

Zadanie 1c: W komórce Markdown odpowiedz: dlaczego inner ma mniej wierszy niż left? Wyjaśnij w 1-2
zdaniach

Inner (złączenie wewnętrzne) wymaga pełnego dopasowania kluczy po obu stronach. Książka testowa z autor_id = 999 nie ma pasującego odpowiednika w tabeli z autorami, więc inner całkowicie ją pomija. Z kolei left (złączenie lewostronne) zachowuje ten wiersz, wstawiając wartości NaN w brakujących kolumnach pobieranych z prawej tabeli.

Ćwiczenie 2: Merge łańcuchowy + kolumny wyliczane (20 min)

In [6]:
# Połącz: zamowienia + ksiazki + autorzy
pelne = (
    zamowienia
    .merge(ksiazki, on='ksiazka_id')
    .merge(autorzy, on='autor_id')
)
print(f"Pelna tabela: {pelne.shape}")
pelne.head()

Pelna tabela: (80, 15)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
1,5002,107,2,2026-02-08,web,Poznań,4,Wiersze wybrane,poezja,35.0,180,Wisława,Szymborska,Polska,Nobel
2,5003,111,1,2026-04-19,web,Poznań,7,Król,historyczna,59.0,688,Szczepan,Twardoch,Polska,NIKE
3,5004,109,4,2026-02-24,web,Warszawa,6,Wojna polsko-ruska,obyczajowa,38.0,153,Dorota,Masłowska,Polska,Polityka
4,5005,105,2,2026-01-10,web,Warszawa,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


Zadanie 2a: Dodaj cztery nowe kolumny:

In [7]:
# 1. Wartość zamówienia 
pelne['wartosc'] = pelne['ilosc'] * pelne['cena'] 
# 2. Miesiąc zamówienia 
pelne['miesiac'] = pelne['data'].dt.month 
# 3. Pełne imię i nazwisko autora 
pelne['autor_pelne'] = pelne['imie'] + " " + pelne['nazwisko'] 
# 4. Kategoria cenowa książki 
pelne['kategoria_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania') 

print(f"Pelna tabela: {pelne.shape}") 
pelne.head() 

Pelna tabela: (80, 19)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody,wartosc,miesiac,autor_pelne,kategoria_cenowa
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel,245.0,4,Olga Tokarczuk,tania
1,5002,107,2,2026-02-08,web,Poznań,4,Wiersze wybrane,poezja,35.0,180,Wisława,Szymborska,Polska,Nobel,70.0,2,Wisława Szymborska,tania
2,5003,111,1,2026-04-19,web,Poznań,7,Król,historyczna,59.0,688,Szczepan,Twardoch,Polska,NIKE,59.0,4,Szczepan Twardoch,droga
3,5004,109,4,2026-02-24,web,Warszawa,6,Wojna polsko-ruska,obyczajowa,38.0,153,Dorota,Masłowska,Polska,Polityka,152.0,2,Dorota Masłowska,tania
4,5005,105,2,2026-01-10,web,Warszawa,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF,90.0,1,Andrzej Sapkowski,tania


Zadanie 2b: Odpowiedz na pytania:

In [8]:
# 1. Ile jest wszystkich zamówień?
print(f"1. Ile jest wszystkich zamówień? {pelne['zam_id'].nunique()}")
# 2. Jaki jest łączny przychód?
print(f"2. Jaki jest łączny przychód? {pelne['wartosc'].sum():.2f} zł") 
# 3. Jaka jest średnia wartość zamówienia?
print(f"3. Jaka jest średnia wartość zamówienia? {pelne['wartosc'].mean():.2f} zł") 
# 4. Ile książek (unikalnych tytułów) się sprzedało? (nunique)
print(f"4. Ile książek (unikalnych tytułów) się sprzedało? {pelne['tytul'].nunique()}") 

1. Ile jest wszystkich zamówień? 80
2. Jaki jest łączny przychód? 11963.80 zł
3. Jaka jest średnia wartość zamówienia? 149.55 zł
4. Ile książek (unikalnych tytułów) się sprzedało? 12


Ćwiczenie 3: groupby + .agg() — raporty per kategoria (30 min)

Zadanie 3a: Dla każdej kategoria policz łączny przychód (suma wartosc) i posortuj malejąco:

In [9]:
pelne.groupby('kategoria')['wartosc'].sum().sort_values(ascending=False)

kategoria
historyczna    4136.8
sci-fi         2334.0
obyczajowa     1999.0
fantasy        1930.0
reportaz       1144.0
poezja          420.0
Name: wartosc, dtype: float64

Zadanie 3b: Dla każdej kategoria policz: liczbę zamówień, średnią wartość, łączną wartość:

In [10]:
pelne.groupby('kategoria')['wartosc'].agg(['count', 'mean', 'sum']).round(2)

,count,mean,sum
kategoria,,,
fantasy,14,137.86,1930.0
historyczna,22,188.04,4136.8
obyczajowa,17,117.59,1999.0
poezja,7,60.00,420.0
reportaz,5,228.80,1144.0
sci-fi,15,155.60,2334.0


Zadanie 3c: Zbuduj raport per autor używając named aggregation.

In [13]:
raport_autorzy = pelne.groupby('autor_pelne').agg( 
    liczba_zamowien=('zam_id', 'count'), 
    laczna_sprzedaz=('wartosc', 'sum'), 
    sredni_ilosc=('ilosc', 'mean') 
).round(2).sort_values('laczna_sprzedaz', ascending=False)

Zadanie 3d: Policz łączną wartość zamówień per kategoria × kanal:

In [14]:
pelne.groupby(['kategoria', 'kanal'])['wartosc'].sum()

kategoria    kanal    
fantasy      aplikacja     475.0
             telefon       225.0
             web          1230.0
historyczna  aplikacja    1988.5
             telefon       527.4
             web          1620.9
obyczajowa   aplikacja     586.0
             telefon       136.0
             web          1277.0
poezja       aplikacja      70.0
             telefon        70.0
             web           280.0
reportaz     aplikacja     416.0
             telefon       260.0
             web           468.0
sci-fi       aplikacja     534.0
             telefon       279.0
             web          1521.0
Name: wartosc, dtype: float64

Zadanie 3e: Dodaj do pelne kolumnę udzial_w_kategorii_pct:

In [15]:
pelne['wartosc_kat'] = pelne.groupby('kategoria')['wartosc'].transform('sum') 
pelne['udzial_w_kategorii_pct'] = (pelne['wartosc'] / pelne['wartosc_kat'] * 100).round(2) 

pelne.groupby('kategoria')['udzial_w_kategorii_pct'].sum()

kategoria
fantasy         99.98
historyczna    100.00
obyczajowa      99.98
poezja         100.00
reportaz       100.01
sci-fi         100.00
Name: udzial_w_kategorii_pct, dtype: float64

Ćwiczenie 4: pivot_table + crosstab — raport biznesowy (20 min)

Zadanie 4a: Zbuduj tabelę przestawną z łącznym przychodem:
Zadanie 4b: Dodaj margins=True, margins_name='RAZEM'

In [16]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac', 
               values='wartosc', aggfunc='sum', fill_value=0, 
               margins=True, margins_name='RAZEM')

miesiac,1,2,3,4,RAZEM
kategoria,,,,,
fantasy,595.0,410.0,270.0,655.0,1930.0
historyczna,1342.5,1790.9,634.4,369.0,4136.8
obyczajowa,301.0,623.0,274.0,801.0,1999.0
poezja,105.0,70.0,140.0,105.0,420.0
reportaz,416.0,0.0,728.0,0.0,1144.0
sci-fi,765.0,615.0,672.0,282.0,2334.0
RAZEM,3524.5,3508.9,2718.4,2212.0,11963.8


Zadanie 4c:

In [17]:
pd.crosstab(pelne['kanal'], pelne['miasto'])

miasto,Gdańsk,Kraków,Poznań,Warszawa,Wrocław
kanal,,,,,
aplikacja,7,1,5,6,4
telefon,1,4,1,3,2
web,5,9,14,9,9


Zadanie 4d: Przekształć w rozkład procentowy w wierszu:

In [18]:
(pd.crosstab(pelne['kanal'], pelne['miasto'], normalize='index') * 100).round(2)

miasto,Gdańsk,Kraków,Poznań,Warszawa,Wrocław
kanal,,,,,
aplikacja,30.43,4.35,21.74,26.09,17.39
telefon,9.09,36.36,9.09,27.27,18.18
web,10.87,19.57,30.43,19.57,19.57


Zadanie 4e: W komórce Markdown napisz 3-5 zdań — wnioski z twojej analizy. Które kategorie, miesiące,
kanały dominują? Co by powiedział kierownik sprzedaży otwierając twój notebook?

Kategoria książek historycznych wygenerowała największy łączny przychód, dominując w zestawieniu (TOP 2 to zwykle książki historyczne i obyczajowe). Jeśli chodzi o kanały dystrybucji, kanał "web" odpowiada za największy odsetek spływających zamówień w poszczególnych miastach. Podsumowując, działania promocyjne w kolejnych kwartałach powinny w pierwszej kolejności skupić się na najsilniejszych kategoriach (historyczna/obyczajowa) z naciskiem na platformę sprzedażową WWW.